# Automatic labelling of Urdu notice pages (Surya + metadata)

This notebook labels every page **without drawing boxes by hand**:

1. **Surya** finds the layout blocks and reads every text line (OCR).
2. The blocks are mapped to our labels: `header, title, body, table, reference, signature`.
3. The fields `date, department, subject, sro_number` are found by matching OCR lines with `pages.csv`.
4. Outputs: `auto_annotations.json`, YOLO labels, preview images and `review.csv`, saved to Google Drive.

**Colab:** Runtime → Change runtime type → **T4 GPU**, then run the cells from top to bottom.
**Your own PC:** start Jupyter from the environment that has Surya and CUDA PyTorch installed, set `PROJECT` in cell 1, then run the cells.

These are automatic ("silver") labels. Use them for training. For the **test** pages, look at the previews and mark each page OK or wrong in `review.csv` (step 9).

## 1. Settings: change the folder path if yours is different

In [ ]:
import os, sys
IN_COLAB = False
PROJECT    = r"T:\nastaliq"
IMAGES_DIR = PROJECT + r"\pages\annotate_images"
PAGES_CSV  = PROJECT + r"\pages\manifest.csv"
OUT_DIR    = PROJECT + r"\auto_labels"
ONLY_SPLIT = ""
MAX_PAGES  = 20
CHUNK      = 100
WORK       = os.path.join(PROJECT, "work")

## 2. Connect Google Drive and check the files

In [ ]:
import os, csv, subprocess
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

assert os.path.isdir(IMAGES_DIR), "Image folder not found: " + IMAGES_DIR
imgs = sorted(n for n in os.listdir(IMAGES_DIR) if n.lower().endswith((".png", ".jpg", ".jpeg")))
print("images found:", len(imgs))
print("pages.csv found:", os.path.exists(PAGES_CSV))
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
      or "NO GPU: Runtime -> Change runtime type -> T4 GPU")
os.makedirs(OUT_DIR, exist_ok=True)

## 3. Install Surya (Colab only; on your PC it is installed before starting Jupyter)

In [ ]:
if IN_COLAB:
    !pip -q install surya-ocr
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU, will be very slow")

## 4. Pick the pages and split them into chunks

In [ ]:
import shutil
split_of, meta = {}, {}
if os.path.exists(PAGES_CSV):
    with open(PAGES_CSV, encoding="utf-8-sig") as f:
        for r in csv.DictReader(f):
            stem = os.path.splitext(os.path.basename(r["image"]))[0]
            split_of[stem] = r.get("split", "")

chosen = [n for n in imgs if not ONLY_SPLIT or split_of.get(os.path.splitext(n)[0]) == ONLY_SPLIT]
if MAX_PAGES:
    chosen = chosen[:MAX_PAGES]
print("pages to label:", len(chosen))

shutil.rmtree(WORK, ignore_errors=True)
chunks = []
for i in range(0, len(chosen), CHUNK):
    d = os.path.join(WORK, "chunk_%03d" % (i // CHUNK))
    os.makedirs(d)
    for n in chosen[i:i + CHUNK]:
        shutil.copy(os.path.join(IMAGES_DIR, n), d)      # local copy: much faster than reading from Drive
    chunks.append(d)
print("chunks:", len(chunks))

## 5. Run Surya layout and OCR on every chunk

On a T4 GPU expect very roughly a few seconds per page. Results go straight to Drive, so if Colab disconnects, run cells 2, 3, 4 and this cell again: finished chunks are skipped.

In [ ]:
import glob, json, time, surya

# small batches so it fits a 6 GB GPU (GTX 1660 Super) as well as Colab's T4
os.environ.setdefault("LAYOUT_BATCH_SIZE", "4")
os.environ.setdefault("DETECTOR_BATCH_SIZE", "4")
os.environ.setdefault("RECOGNITION_BATCH_SIZE", "32")
RAW = os.path.join(OUT_DIR, "surya_raw")
SITE = os.path.dirname(os.path.dirname(surya.__file__))
BIN = os.path.dirname(sys.executable)

def exe(cmd):
    """Find the Surya command next to this Python (works inside a venv on Windows too)."""
    for c in (os.path.join(BIN, cmd + ".exe"), os.path.join(BIN, "Scripts", cmd + ".exe"), os.path.join(BIN, cmd)):
        if os.path.exists(c):
            return c
    return shutil.which(cmd) or cmd

def has_json(d):
    return bool(glob.glob(os.path.join(d, "**", "*.json"), recursive=True))

def run_surya(cmd, data_dir, out_dir, extra=()):
    """Run a Surya command; works whether or not this Surya version supports --output_dir."""
    start = time.time()
    os.makedirs(out_dir, exist_ok=True)
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    r = subprocess.run([exe(cmd), data_dir, "--output_dir", out_dir, *extra], capture_output=True, text=True,
                       encoding="utf-8", errors="replace", env=env)
    if r.returncode != 0 and "output_dir" in (r.stderr + r.stdout):
        r = subprocess.run([exe(cmd), data_dir, *extra], capture_output=True, text=True,
                           encoding="utf-8", errors="replace", env=env)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-3000:])
        raise RuntimeError(cmd + " failed (see the message above)")
    if not has_json(out_dir):   # some versions write to their own results folder: collect it
        for root in (WORK, os.getcwd(), SITE):
            for p in glob.glob(os.path.join(root, "**", "results.json"), recursive=True):
                if os.path.getmtime(p) >= start and not p.startswith(RAW):
                    shutil.copytree(os.path.dirname(p), os.path.join(out_dir, "found"), dirs_exist_ok=True)
    if not has_json(out_dir):
        print((r.stdout + r.stderr)[-2000:])
        raise RuntimeError("no results.json found for " + cmd)

for d in chunks:
    name = os.path.basename(d)
    lay_out, ocr_out = os.path.join(RAW, "layout", name), os.path.join(RAW, "ocr", name)
    t = time.time()
    if not has_json(lay_out):
        run_surya("surya_layout", d, lay_out)
    if not has_json(ocr_out):
        run_surya("surya_ocr", d, ocr_out)
    print("%s done (%d pages, %.0f s)" % (name, len(os.listdir(d)), time.time() - t))
print("all chunks finished")

## 6. Conversion code (Surya output → our labels)

Just run this cell; it only defines functions.

In [ ]:
%%writefile convert.py
import csv, glob, json, os, re, difflib
import numpy as np
from PIL import Image, ImageDraw, ImageFont

try:
    FONT = ImageFont.load_default(size=26)
except Exception:
    FONT = None

LAYOUT = ["header", "title", "body", "table", "reference", "signature"]
FIELDS = ["date", "department", "subject", "sro_number"]

# Surya layout label -> our label (None = ignore). Keys are lower-case with non-letters removed.
SURYA_MAP = {
    "pageheader": "header", "sectionheader": "title", "title": "title",
    "text": "body", "listitem": "body", "caption": "body", "footnote": "body", "textinlinemath": "body",
    "equation": "body", "formula": "body", "code": "body", "handwriting": "signature",
    "table": "table", "form": "table", "tableofcontents": "table",
    "pagefooter": None, "picture": "picture", "figure": "picture",
}
URDU_MONTHS = ["جنوری", "فروری", "مارچ", "اپریل", "مئی", "جون", "جولائی", "اگست", "ستمبر", "اکتوبر", "نومبر", "دسمبر"]
EN_MONTHS = ["january", "february", "march", "april", "may", "june", "july", "august", "september", "october", "november", "december"]
DEPT_WORDS = ["ڈویژن", "ڈویزن", "وزارت", "بورڈ", "division", "ministry", "board"]
SUBJ_WORDS = ["کسٹمز", "سیلز ٹیکس", "انکم ٹیکس", "ایکسائز", "customs", "sales tax", "income tax", "excise",
              "پریس ریلیز", "پریس ریلز", "نوٹیفکیشن", "عنوان", "بابت", "press release"]
TITLE_WORDS = ["نوٹیفکیشن", "نوٹیفیکیشن", "اعلامیہ", "notification", "پریس ریلیز", "پریس ریلز", "press release"]
SIGN_WORDS = ["دستخط", "سیکرٹری", "سیکریٹری", "secretary", "چیئرمین", "ممبر", "ڈائریکٹر"]
REF_RE = re.compile(r"\b[CF]\s*\.?\s*No\b|^\s*\[|\]\s*$|نمبر\s*بورڈ|نمبر\s*/|No\.\s*\d", re.I)
DATE_NUM_RE = re.compile(r"\b\d{1,2}\s*[-/.]\s*\d{1,2}\s*[-/.]\s*(19|20)?\d{2}\b")
URDU_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def norm_label(s):
    return re.sub(r"[^a-z]", "", str(s).lower())


def digits(s):
    return str(s).translate(URDU_DIGITS)


def find_results(root):
    """All results.json files under root -> merged {image_stem: page_dict}."""
    out = {}
    for path in glob.glob(os.path.join(root, "**", "*.json"), recursive=True):
        try:
            data = json.load(open(path, encoding="utf-8"))
        except Exception:
            continue
        if not isinstance(data, dict):
            continue
        for key, val in data.items():
            pages = val if isinstance(val, list) else [val]
            if pages and isinstance(pages[0], dict):
                out[os.path.splitext(os.path.basename(key))[0]] = pages[0]
    return out


def scale_for(page, W, H):
    ib = page.get("image_bbox")
    if ib and len(ib) == 4 and ib[2] > 0 and ib[3] > 0:
        return W / float(ib[2] - ib[0]), H / float(ib[3] - ib[1])
    return 1.0, 1.0


def to_box(item, sx, sy):
    b = item.get("bbox")
    if not b and item.get("polygon"):
        xs = [p[0] for p in item["polygon"]]
        ys = [p[1] for p in item["polygon"]]
        b = [min(xs), min(ys), max(xs), max(ys)]
    if not b:
        return None
    return [b[0] * sx, b[1] * sy, b[2] * sx, b[3] * sy]


def layout_items(page, W, H):
    sx, sy = scale_for(page, W, H)
    items = []
    for it in page.get("bboxes", []) or []:
        box = to_box(it, sx, sy)
        if box:
            items.append({"surya": it.get("label", ""), "box": box, "pos": it.get("position", 0)})
    return items


def ocr_lines(page, W, H):
    sx, sy = scale_for(page, W, H)
    lines = []
    for it in page.get("text_lines", []) or []:
        box = to_box(it, sx, sy)
        if box:
            lines.append({"text": (it.get("text") or "").strip(), "box": box, "conf": it.get("confidence", None)})
    lines.sort(key=lambda l: (l["box"][1], -l["box"][2]))
    return lines


def overlap_frac(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    area = max(1e-6, (a[2] - a[0]) * (a[3] - a[1]))
    return ix * iy / area


def lines_in(box, lines):
    return [l for l in lines if overlap_frac(l["box"], box) > 0.5]


def has(text, words):
    t = text.lower()
    return any(w in t for w in words)


def assign_layout(items, lines, W, H):
    """Surya blocks -> our six labels, using Surya's label, the text inside, and position."""
    blocks = []
    for it in items:
        lab = SURYA_MAP.get(norm_label(it["surya"]), "body")
        if lab is None:
            continue
        txt = " ".join(l["text"] for l in lines_in(it["box"], lines))
        blocks.append({"label": lab, "box": it["box"], "text": txt, "surya": it["surya"]})
    blocks.sort(key=lambda b: (b["box"][1], -b["box"][2]))

    # title: keyword beats Surya's guess
    title_i = next((i for i, b in enumerate(blocks) if has(b["text"], TITLE_WORDS)), None)
    if title_i is None:
        title_i = next((i for i, b in enumerate(blocks) if b["label"] == "title" and b["box"][1] < H * 0.45), None)
    for i, b in enumerate(blocks):
        if b["label"] == "title" and i != title_i:
            b["label"] = "body"
    if title_i is not None:
        blocks[title_i]["label"] = "title"
        for i in range(title_i):
            if blocks[i]["label"] in ("body", "header"):
                blocks[i]["label"] = "header"
    # reference line
    ref_i = None
    for i, b in enumerate(blocks):
        if b["label"] != "table" and b["box"][1] > H * 0.3 and REF_RE.search(b["text"]) and len(b["text"]) < 80:
            b["label"] = "reference"
            ref_i = i
    # signature: picture/handwriting near bottom, signature words, or blocks after the reference
    for i, b in enumerate(blocks):
        bottom = b["box"][1] > H * 0.55
        if b["label"] == "picture":
            b["label"] = "signature" if bottom else None
        elif bottom and b["label"] in ("body", "header") and has(b["text"], SIGN_WORDS) and len(b["text"]) < 150:
            b["label"] = "signature"
        elif ref_i is not None and i > ref_i and b["label"] == "body" and (b["box"][2] - b["box"][0]) < W * 0.5:
            b["label"] = "signature"
    return [b for b in blocks if b["label"]]


def best_line(lines, test, region=None):
    for l in lines:
        if region and overlap_frac(l["box"], region) < 0.5:
            continue
        if test(l["text"]):
            return l
    return None


def fuzzy(a, b):
    return difflib.SequenceMatcher(None, digits(a), digits(b)).ratio()


def find_fields(blocks, lines, meta, first_page, W, H):
    fields = []
    head = [b["box"] for b in blocks if b["label"] in ("header", "title")]
    top = [0, 0, W, H * 0.45]
    sro_val = meta.get("sro_number", "")
    sro_num = re.findall(r"\d{2,4}", digits(sro_val))
    date_val = meta.get("date", "")
    title_val = meta.get("title", "")

    # SRO number: line containing the number from metadata, else an S.R.O pattern
    l = None
    if sro_num:
        l = best_line(lines, lambda t: sro_num[0] in digits(t) and (has(t, ["s.r.o", "sro", "ایس آر او", "o.r.s"]) or "(" in t))
    if l is None:
        l = best_line(lines, lambda t: bool(re.search(r"S\s*\.?\s*R\s*\.?\s*O|O\s*\.?\s*R\s*\.?\s*S|ایس\s*آر\s*او", t, re.I)), top)
    if l is not None:
        fields.append(("sro_number", l["box"], sro_val if (first_page and sro_val) else l["text"], "match" if sro_num else "pattern"))

    # date: metadata fuzzy match, else month name + year in the top part
    l = None
    if date_val:
        cands = [(fuzzy(x["text"], date_val), x) for x in lines if x["box"][1] < H * 0.5]
        cands.sort(key=lambda c: -c[0])
        if cands and cands[0][0] > 0.45:
            l = cands[0][1]
    if l is None:
        l = best_line(lines, lambda t: has(t, URDU_MONTHS + EN_MONTHS) and re.search(r"(19|20)\d\d", digits(t)), top)
    if l is None:
        l = best_line(lines, lambda t: "مورخہ" in t or bool(DATE_NUM_RE.search(digits(t))))
    if l is not None:
        fields.append(("date", l["box"], date_val if (first_page and date_val and l) else l["text"], "match" if date_val else "pattern"))

    # reference number: SRO number above covers FBR; AJK and others use "نمبر بورڈ/..." or "C.No."
    if not any(f[0] == "sro_number" for f in fields):
        l = best_line(lines, lambda t: bool(REF_RE.search(t)) and bool(re.search(r"\d", digits(t))))
        if l is not None:
            fields.append(("sro_number", l["box"], l["text"], "pattern"))

    # department: header line with a department word (not the "Government of Pakistan" line)
    l = best_line(lines, lambda t: has(t, DEPT_WORDS) and not has(t, ["حکومت پاکستان", "government of pakistan"]), top)
    if l is not None:
        fields.append(("department", l["box"], l["text"], "pattern"))

    # subject: line with a tax/customs word inside the title area, else fuzzy match with the title
    title_boxes = [b["box"] for b in blocks if b["label"] == "title"]
    l = None
    for tb in title_boxes:
        l = best_line(lines, lambda t: has(t, SUBJ_WORDS), [tb[0], tb[1], tb[2], tb[3] + H * 0.05])
        if l:
            break
    if l is None:
        l = best_line(lines, lambda t: has(t, SUBJ_WORDS) and len(t) < 40, top)
    if l is None and title_val:
        cands = sorted(((fuzzy(x["text"], title_val), x) for x in lines), key=lambda c: -c[0])
        if cands and cands[0][0] > 0.5:
            l = cands[0][1]
    if l is not None:
        fields.append(("subject", l["box"], l["text"], "pattern"))
    return fields


COLORS = {"header": (30, 120, 180), "title": (30, 160, 30), "body": (120, 120, 120), "table": (160, 60, 160),
          "reference": (220, 120, 30), "signature": (200, 30, 30), "date": (255, 140, 0),
          "department": (0, 180, 180), "subject": (180, 180, 0), "sro_number": (255, 0, 255)}


def draw_preview(img, blocks, fields, path, max_h=1400):
    vis = img.convert("RGB").copy()
    d = ImageDraw.Draw(vis)
    for b in blocks:
        d.rectangle(b["box"], outline=COLORS[b["label"]], width=4)
        d.text((b["box"][0] + 4, max(0, b["box"][1] - 30)), b["label"], fill=COLORS[b["label"]], font=FONT)
    for name, box, _, _ in fields:
        d.rectangle(box, outline=COLORS[name], width=3)
        d.text((box[0] + 4, box[3] + 2), name, fill=COLORS[name], font=FONT)
    if vis.height > max_h:
        vis = vis.resize((int(vis.width * max_h / vis.height), max_h))
    vis.save(path, quality=85)


def convert(images_dir, layout_root, ocr_root, pages_csv, out_dir, preview_split="test", max_previews=400):
    meta = {}
    if pages_csv and os.path.exists(pages_csv):
        with open(pages_csv, encoding="utf-8-sig") as f:
            for r in csv.DictReader(f):
                meta[os.path.splitext(os.path.basename(r["image"]))[0]] = r
    lay = find_results(layout_root)
    ocr = find_results(ocr_root)
    print("Surya layout results:", len(lay), "| OCR results:", len(ocr))
    os.makedirs(os.path.join(out_dir, "yolo", "labels"), exist_ok=True)
    os.makedirs(os.path.join(out_dir, "preview"), exist_ok=True)
    open(os.path.join(out_dir, "yolo", "classes.txt"), "w").write("\n".join(LAYOUT) + "\n")

    names = sorted(n for n in os.listdir(images_dir) if n.lower().endswith((".png", ".jpg", ".jpeg")))
    all_pages, counts, review = [], {k: 0 for k in LAYOUT + FIELDS}, []
    n_prev = 0
    missing = 0
    for name in names:
        stem = os.path.splitext(name)[0]
        if stem not in lay:
            missing += 1
            continue
        img = Image.open(os.path.join(images_dir, name))
        W, H = img.size
        m = meta.get(stem, {})
        first = str(m.get("page", "1")) == "1"
        lines = ocr_lines(ocr.get(stem, {}), W, H)
        blocks = assign_layout(layout_items(lay[stem], W, H), lines, W, H)
        fields = find_fields(blocks, lines, m, first, W, H)
        regions = [{"kind": "layout", "label": b["label"], "bbox": [round(v, 1) for v in b["box"]], "text": b["text"],
                    "surya_label": b["surya"]} for b in blocks]
        regions += [{"kind": "field", "label": f[0], "bbox": [round(v, 1) for v in f[1]], "text": f[2], "found_by": f[3]}
                    for f in fields]
        regions += [{"kind": "line", "label": "line", "bbox": [round(v, 1) for v in l["box"]], "text": l["text"]}
                    for l in lines]
        for b in blocks:
            counts[b["label"]] += 1
        for f in fields:
            counts[f[0]] += 1
        all_pages.append({"image": name, "width": W, "height": H, "split": m.get("split", ""),
                          "doc_id": m.get("doc_id", ""), "page": m.get("page", ""), "regions": regions})
        with open(os.path.join(out_dir, "yolo", "labels", stem + ".txt"), "w") as f:
            for b in blocks:
                x0, y0, x1, y1 = b["box"]
                f.write("%d %.6f %.6f %.6f %.6f\n" % (LAYOUT.index(b["label"]), (x0 + x1) / 2 / W,
                                                      (y0 + y1) / 2 / H, (x1 - x0) / W, (y1 - y0) / H))
        if (not preview_split or m.get("split") == preview_split) and n_prev < max_previews:
            draw_preview(img, blocks, fields, os.path.join(out_dir, "preview", stem + ".jpg"))
            n_prev += 1
            found = {f[0] for f in fields}
            review.append({"image": name, "preview": "preview/" + stem + ".jpg", "split": m.get("split", ""),
                           "page": m.get("page", ""), "fields_found": " ".join(sorted(found)),
                           "fields_missing": " ".join(x for x in FIELDS if x not in found) if first else "",
                           "status": "", "note": ""})
    json.dump(all_pages, open(os.path.join(out_dir, "auto_annotations.json"), "w", encoding="utf-8"),
              ensure_ascii=False, indent=1)
    with open(os.path.join(out_dir, "review.csv"), "w", encoding="utf-8-sig", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["image", "preview", "split", "page", "fields_found", "fields_missing", "status", "note"])
        w.writeheader()
        w.writerows(review)
    print("pages labelled:", len(all_pages), "| pages without Surya output:", missing)
    print("boxes:", counts)
    print("previews:", n_prev, "->", os.path.join(out_dir, "preview"))
    return all_pages

## 7. Make the labels

In [ ]:
import sys, importlib
sys.path.insert(0, os.getcwd())
import convert; importlib.reload(convert)

pages = convert.convert(
    images_dir=IMAGES_DIR,
    layout_root=os.path.join(RAW, "layout"),
    ocr_root=os.path.join(RAW, "ocr"),
    pages_csv=PAGES_CSV,
    out_dir=OUT_DIR,
    preview_split="",          # previews + review rows for the test pages ("" = every page)
)

## 8. Look at a few previews

In [ ]:
from IPython.display import Image as Show, display
prev = sorted(glob.glob(os.path.join(OUT_DIR, "preview", "*.jpg")))
print(len(prev), "preview images in", os.path.join(OUT_DIR, "preview"))
for p in prev[:4]:
    print(os.path.basename(p)); display(Show(filename=p, width=650))

## 9. Quick check of the test pages (the only manual step)

Open `auto_labels/review.csv` (Drive → open with Google Sheets) and `auto_labels/preview/` side by side.
For each row, look at the preview and type in **status**:

- `ok` if the boxes and fields look right
- `wrong` if something important is off, and say what in **note**

This is looking, not drawing: about 30 seconds a page. Report test scores only on pages marked `ok`, or fix the `wrong` ones.

## 10. Build the YOLO dataset for the layout model

In [ ]:
YOLO = os.path.join(OUT_DIR, "yolo_dataset")
lab_dir = os.path.join(OUT_DIR, "yolo", "labels")
split_name = {"train": "train", "dev": "val", "test": "test"}
n = {}
for p in pages:
    stem = os.path.splitext(p["image"])[0]
    s = split_name.get(p["split"], "train")
    for sub in ("images", "labels"):
        os.makedirs(os.path.join(YOLO, sub, s), exist_ok=True)
    shutil.copy(os.path.join(IMAGES_DIR, p["image"]), os.path.join(YOLO, "images", s, p["image"]))
    shutil.copy(os.path.join(lab_dir, stem + ".txt"), os.path.join(YOLO, "labels", s, stem + ".txt"))
    n[s] = n.get(s, 0) + 1
with open(os.path.join(YOLO, "data.yaml"), "w") as f:
    f.write("path: %s\ntrain: images/train\nval: images/val\ntest: images/test\n" % YOLO)
    f.write("names: %s\n" % convert.LAYOUT)
print("YOLO dataset:", n, "->", YOLO)

## Outputs (in Drive → `auto_labels/`)

| File | What it is |
|---|---|
| `auto_annotations.json` | every page: layout boxes, field boxes (with values) and all OCR text lines with boxes |
| `yolo/labels/*.txt`, `yolo/classes.txt` | layout boxes in YOLO format |
| `yolo_dataset/` + `data.yaml` | ready for `yolo train data=.../data.yaml` |
| `preview/*.jpg` | test pages with boxes drawn, for the quick check |
| `review.csv` | one row per test page; fill in `status` |
| `surya_raw/` | Surya's raw output, kept so you never need to rerun it |